In [1]:
import pandas as pd
from pathlib import Path
import re

# Point to the raw data folder (we're in analysis/, so go up one level)
RAW_DIR = Path('../data/raw')

# List every file in raw/
files = sorted(RAW_DIR.glob('*.xls*'))
print(f"Found {len(files)} files:\n")
for f in files:
    print(f"  {f.name}")
    

Found 16 files:

  20194YearGradRateFinal.xls
  4Y-GradRate-Cohort2024.xlsx
  4Y-GradRate-Cohort2025.xlsx
  4Year_Grad_Rate_Cohort2023_publish[1] (1).xlsx
  4Ygraduation_rate_cohort_2022_redacted.xlsx
  Cohort 2010_4 Year Graduation Rate.xls
  Cohort 2011_4 Year Graduation Rate.xls
  Cohort 2012_4 Year Graduation Rate.xls
  Cohort 2013_4 Year Graduation Rate.xls
  Cohort 2014_4 Year Graduation Rate.xls
  Cohort 2015_4 Year Graduation Rate.xls
  Cohort 2016_4 Year Graduation Rate.xls
  Cohort 2017_4 Year Graduation Rate.xls
  Cohort 2018 Four Year_Redacted.xls
  Cohort 2020 4 Year Graduation Rate Final.xlsx
  Cohort 2021 4 Year Graduation Rate.xlsx


In [2]:
def extract_cohort_year(filename):
    """Pull the 4-digit cohort year out of any ADE filename."""
    match = re.search(r'(20\d{2})', filename)
    return int(match.group(1)) if match else None

# For each file, report: cohort year, sheet names, dimensions of first sheet
for f in files:
    year = extract_cohort_year(f.name)
    try:
        xl = pd.ExcelFile(f)
        sheets = xl.sheet_names
        # Read first sheet to get dimensions
        df = pd.read_excel(f, sheet_name=0, header=None)
        print(f"Cohort {year}  |  {f.name}")
        print(f"  Sheets ({len(sheets)}): {sheets}")
        print(f"  First sheet shape: {df.shape[0]} rows x {df.shape[1]} cols")
        print()
    except Exception as e:
        print(f"ERROR reading {f.name}: {e}\n")

Cohort 2019  |  20194YearGradRateFinal.xls
  Sheets (5): ['Notes', 'School by Subgroup', 'LEA by Subgroup', 'County by Subgroup', 'State by Subgroup']
  First sheet shape: 0 rows x 0 cols

Cohort 2024  |  4Y-GradRate-Cohort2024.xlsx
  Sheets (7): ['Introduction', 'School by Subgroup', 'LEA by Subgroup', 'County by Subgroup', 'State by Subgroup', 'Data Dictionary', 'Subgroup Dictionary']
  First sheet shape: 10 rows x 1 cols

Cohort 2025  |  4Y-GradRate-Cohort2025.xlsx
  Sheets (7): ['Introduction', 'School by Subgroup', 'LEA by Subgroup', 'County by Subgroup', 'State by Subgroup', 'Data Dictionary', 'Subgroup Dictionary']
  First sheet shape: 10 rows x 1 cols

Cohort 2023  |  4Year_Grad_Rate_Cohort2023_publish[1] (1).xlsx
  Sheets (7): ['Report Introduction', 'School by Subgroup', 'LEA by Subgroup', 'County by Subgroup', 'State by Subgroup', 'Subgroup Dictionary', 'Data Dictionary']
  First sheet shape: 0 rows x 0 cols

Cohort 2022  |  4Ygraduation_rate_cohort_2022_redacted.xlsx
  Shee

In [3]:
def get_data_sheet_name(sheet_names, level='school'):
    """Find the right data sheet across both format eras."""
    # New era: 'School by Subgroup', 'LEA by Subgroup', etc.
    # Old era: 'School', 'LEA', 'County', 'State'
    target_new = f'{level.capitalize()} by Subgroup' if level != 'lea' else 'LEA by Subgroup'
    target_old = level.capitalize() if level != 'lea' else 'LEA'
    
    for name in sheet_names:
        if name == target_new:
            return name
    for name in sheet_names:
        if name == target_old:
            return name
    return None

# Pick three representative years to inspect in detail
samples = [2015, 2018, 2025]

for year in samples:
    # Find the file matching this year
    f = next((f for f in files if extract_cohort_year(f.name) == year), None)
    if not f:
        continue
    
    xl = pd.ExcelFile(f)
    sheet = get_data_sheet_name(xl.sheet_names, 'school')
    
    print(f"{'='*70}")
    print(f"Cohort {year}  |  Sheet: '{sheet}'")
    print(f"{'='*70}")
    
    # Read first 10 rows with NO header parsing so we see raw structure
    df = pd.read_excel(f, sheet_name=sheet, header=None, nrows=10)
    print(f"Shape (first 10 rows): {df.shape}")
    print("\nRaw first 10 rows:")
    # Show all columns by temporarily expanding display
    with pd.option_context('display.max_columns', None, 'display.width', 200):
        print(df.to_string())
    print("\n")

Cohort 2015  |  Sheet: 'School'
Shape (first 10 rows): (10, 11)

Raw first 10 rows:
            0                     1              2                                     3                 4                               5       6                           7                 8                  9                             10
0  Cohort Year  Graduation Rate Type  LEA Entity ID                              LEA Name  School Entity ID                     School Name  County                    Subgroup  Number Graduated  Number in Cohort   Percent Graduated in 4 Years
1         2015                4 Year          85540  Academy of Building Industries, Inc.             85541  Academy of Building Industries  Mohave                         All                22                 39                         56.41
2         2015                4 Year          85540  Academy of Building Industries, Inc.             85541  Academy of Building Industries  Mohave      Black/African American            

In [4]:
def load_school_data(filepath):
    """Load one file's school-level data, handling both sheet naming conventions."""
    xl = pd.ExcelFile(filepath)
    
    # New era first, then old era
    if 'School by Subgroup' in xl.sheet_names:
        sheet = 'School by Subgroup'
    elif 'School' in xl.sheet_names:
        sheet = 'School'
    else:
        raise ValueError(f"No recognizable school sheet in {filepath.name}")
    
    df = pd.read_excel(filepath, sheet_name=sheet)
    df['source_file'] = filepath.name
    return df

# Load all files and concatenate
all_schools = []
for f in files:
    year = extract_cohort_year(f.name)
    try:
        df = load_school_data(f)
        all_schools.append(df)
        print(f"  Cohort {year}: {len(df):>6,} rows, {df.shape[1]} cols")
    except Exception as e:
        print(f"  Cohort {year}: ERROR — {e}")

schools_raw = pd.concat(all_schools, ignore_index=True, sort=False)
print(f"\nTotal combined: {len(schools_raw):,} rows")
print(f"Columns across all years: {sorted(schools_raw.columns.tolist())}")

  Cohort 2019:  7,902 rows, 12 cols
  Cohort 2024:  6,879 rows, 12 cols
  Cohort 2025:  7,092 rows, 12 cols
  Cohort 2023:  6,847 rows, 12 cols
  Cohort 2022:  8,492 rows, 12 cols
  Cohort 2010:  5,848 rows, 12 cols
  Cohort 2011:  6,104 rows, 12 cols
  Cohort 2012:  6,286 rows, 12 cols
  Cohort 2013:  6,424 rows, 12 cols
  Cohort 2014:  6,999 rows, 12 cols
  Cohort 2015:  6,942 rows, 12 cols
  Cohort 2016:  6,260 rows, 12 cols
  Cohort 2017:  6,327 rows, 12 cols
  Cohort 2018:  6,896 rows, 12 cols
  Cohort 2020:  7,889 rows, 12 cols
  Cohort 2021:  8,078 rows, 12 cols

Total combined: 111,265 rows
Columns across all years: ['Cohort Year', 'County', 'Graduation Rate Type', 'LEA Entity ID', 'LEA Name', 'Number Graduated', 'Number in Cohort', 'Number in Cohort ', 'Percent Graduated', 'Percent Graduated in 4 Years', 'School Entity ID', 'School Name', 'Subgroup', 'source_file']


In [5]:
# Which files have which column name variants?
print("Files with 'Number in Cohort ' (trailing space):")
mask = schools_raw['Number in Cohort '].notna() if 'Number in Cohort ' in schools_raw.columns else None
if mask is not None:
    culprits = schools_raw.loc[mask, 'source_file'].unique()
    for c in culprits:
        print(f"  {c}")

print("\nFiles with 'Percent Graduated' (no '...in 4 Years'):")
if 'Percent Graduated' in schools_raw.columns:
    mask = schools_raw['Percent Graduated'].notna()
    culprits = schools_raw.loc[mask, 'source_file'].unique()
    for c in culprits:
        print(f"  {c}")

print("\nFiles with 'Percent Graduated in 4 Years':")
if 'Percent Graduated in 4 Years' in schools_raw.columns:
    mask = schools_raw['Percent Graduated in 4 Years'].notna()
    culprits = schools_raw.loc[mask, 'source_file'].unique()
    for c in culprits:
        print(f"  {c}")

Files with 'Number in Cohort ' (trailing space):
  Cohort 2015_4 Year Graduation Rate.xls

Files with 'Percent Graduated' (no '...in 4 Years'):
  Cohort 2010_4 Year Graduation Rate.xls

Files with 'Percent Graduated in 4 Years':
  20194YearGradRateFinal.xls
  4Y-GradRate-Cohort2024.xlsx
  4Y-GradRate-Cohort2025.xlsx
  4Year_Grad_Rate_Cohort2023_publish[1] (1).xlsx
  4Ygraduation_rate_cohort_2022_redacted.xlsx
  Cohort 2011_4 Year Graduation Rate.xls
  Cohort 2012_4 Year Graduation Rate.xls
  Cohort 2013_4 Year Graduation Rate.xls
  Cohort 2014_4 Year Graduation Rate.xls
  Cohort 2015_4 Year Graduation Rate.xls
  Cohort 2016_4 Year Graduation Rate.xls
  Cohort 2017_4 Year Graduation Rate.xls
  Cohort 2018 Four Year_Redacted.xls
  Cohort 2020 4 Year Graduation Rate Final.xlsx
  Cohort 2021 4 Year Graduation Rate.xlsx


In [6]:
def load_school_data(filepath):
    """Load one file's school-level data with normalized column names."""
    xl = pd.ExcelFile(filepath)
    
    if 'School by Subgroup' in xl.sheet_names:
        sheet = 'School by Subgroup'
    elif 'School' in xl.sheet_names:
        sheet = 'School'
    else:
        raise ValueError(f"No recognizable school sheet in {filepath.name}")
    
    df = pd.read_excel(filepath, sheet_name=sheet)
    
    # Normalize column names: strip whitespace, standardize the rate column label
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={'Percent Graduated': 'Percent Graduated in 4 Years'})
    
    df['source_file'] = filepath.name
    return df

# Reload everything with the fixed loader
all_schools = []
for f in files:
    df = load_school_data(f)
    all_schools.append(df)

schools_raw = pd.concat(all_schools, ignore_index=True, sort=False)

print(f"Total rows: {len(schools_raw):,}")
print(f"Columns: {schools_raw.columns.tolist()}")
print(f"\nRows per cohort year:")
print(schools_raw.groupby('Cohort Year').size().to_string())

Total rows: 111,265
Columns: ['Cohort Year', 'Graduation Rate Type', 'LEA Entity ID', 'LEA Name', 'School Entity ID', 'School Name', 'County', 'Subgroup', 'Number Graduated', 'Number in Cohort', 'Percent Graduated in 4 Years', 'source_file']

Rows per cohort year:
Cohort Year
2010    5848
2011    6104
2012    6286
2013    6424
2014    6999
2015    6942
2016    6260
2017    6327
2018    6896
2019    7902
2020    7889
2021    8078
2022    8492
2023    6847
2024    6879
2025    7092


In [7]:
def load_level_data(filepath, level):
    """
    Load one file's data at a given aggregation level.
    level must be one of: 'school', 'lea', 'state'
    """
    xl = pd.ExcelFile(filepath)
    
    # Map level to possible sheet names (new era first, then old era)
    sheet_map = {
        'school': ['School by Subgroup', 'School'],
        'lea':    ['LEA by Subgroup', 'LEA'],
        'state':  ['State by Subgroup', 'State'],
    }
    
    sheet = None
    for candidate in sheet_map[level]:
        if candidate in xl.sheet_names:
            sheet = candidate
            break
    
    if sheet is None:
        return None  # Some early files (2010) don't have a County sheet; State/LEA may also be missing
    
    df = pd.read_excel(filepath, sheet_name=sheet)
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={'Percent Graduated': 'Percent Graduated in 4 Years'})
    df['source_file'] = filepath.name
    df['level'] = level
    return df

# Load all three levels
datasets = {'school': [], 'lea': [], 'state': []}

for f in files:
    for level in ['school', 'lea', 'state']:
        df = load_level_data(f, level)
        if df is not None:
            datasets[level].append(df)

schools_raw   = pd.concat(datasets['school'], ignore_index=True, sort=False)
districts_raw = pd.concat(datasets['lea'],    ignore_index=True, sort=False)
statewide_raw = pd.concat(datasets['state'],  ignore_index=True, sort=False)

print(f"Schools:   {len(schools_raw):>7,} rows  ({schools_raw['Cohort Year'].nunique()} years)")
print(f"Districts: {len(districts_raw):>7,} rows  ({districts_raw['Cohort Year'].nunique()} years)")
print(f"Statewide: {len(statewide_raw):>7,} rows  ({statewide_raw['Cohort Year'].nunique()} years)")

print(f"\nStatewide preview (should be ~1 row per year per subgroup):")
print(statewide_raw.head(10).to_string())

Schools:   111,265 rows  (16 years)
Districts:  63,159 rows  (16 years)
Statewide:     250 rows  (16 years)

Statewide preview (should be ~1 row per year per subgroup):
   Cohort Year Graduation Rate Type                          Subgroup Number Graduated Number in Cohort  Percent Graduated in 4 Years                 source_file  level
0         2019               4 Year                               All            68393            86355                         79.20  20194YearGradRateFinal.xls  state
1         2019               4 Year  American Indian or Alaska Native             2617             3772                         69.38  20194YearGradRateFinal.xls  state
2         2019               4 Year                             Asian             2281             2460                         92.72  20194YearGradRateFinal.xls  state
3         2019               4 Year            Black/African American             3482             4615                         75.45  20194YearGradRateFin